In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

# 1. Map sentences to N-gram buckets (Hashing Trick)
# This handles the "Not good" problem by treating "not_good" as its own feature.
def get_features(tokens, vocab_size=20000):
    ngram_hashes = []
    # Unigrams
    ngram_hashes += [hash(t) % vocab_size for t in tokens]
    # Bigrams
    ngram_hashes += [hash(tokens[i] + "_" + tokens[i+1]) % vocab_size for i in range(len(tokens)-1)]
    
    return torch.tensor(list(set(ngram_hashes)), dtype=torch.long)

# 2. Linear Embedding Model (The Math: Shallow Neural Net)
class SentimentNet(nn.Module):
    def __init__(self, vocab_size, embed_dim):
        super().__init__()
        # The Embedding matrix is essentially a lookup table for your learned vectors
        self.embedding = nn.EmbeddingBag(vocab_size, embed_dim, sparse=True)
        self.fc = nn.Linear(embed_dim, 1) # Output a single continuous value for Regression

    def forward(self, x, offsets):
        return self.fc(self.embedding(x, offsets))